# 🌼 Final Project: Flower Species Classification with Transfer Learning

**Dataset**: [TF Flowers (TensorFlow Datasets)](https://www.tensorflow.org/datasets/catalog/tf_flowers)

**Theme**: Image Classification with Transfer Learning  
**Architecture**: Pretrained ResNet18  
**Classes**: 5 flower species  
**Tools**: PyTorch, torchvision, TensorFlow Datasets  


## 📁 1. Dataset Loading and Preprocessing (with TensorFlow Datasets)

In [ ]:
import tensorflow_datasets as tfds
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image

# Download the dataset
dataset, info = tfds.load('tf_flowers', split=['train[:80%]', 'train[80%:]'], with_info=True, as_supervised=True)
train_ds, val_ds = dataset
class_names = info.features['label'].names
num_classes = info.features['label'].num_classes
print("Classes:", class_names)

In [ ]:
# Convert TF dataset to PyTorch-style Dataset
class FlowersDataset(Dataset):
    def __init__(self, tf_ds, transform=None):
        self.data = list(tf_ds)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img, label = self.data[idx]
        img = Image.fromarray(img.numpy())
        if self.transform:
            img = self.transform(img)
        return img, label.numpy()

In [ ]:
# Transforms
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_dataset = FlowersDataset(train_ds, transform=transform_train)
val_dataset = FlowersDataset(val_ds, transform=transform_val)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
classes = class_names

## 🧱 2. Model Design and Implementation

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

model = models.resnet18(pretrained=True)
for param in model.parameters():
    param.requires_grad = False

num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes)

## 🚀 3. Training and Optimization

In [ ]:
from torch.optim import Adam
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.fc.parameters(), lr=0.001)

In [ ]:
def train_model(model, train_loader, val_loader, epochs=5):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader):.4f}")

train_model(model, train_loader, val_loader)

## 📊 4. Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=classes))
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, xticklabels=classes, yticklabels=classes, fmt='d')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()